## 10. Usage Guide

### For Training:
```bash
# Using Python script
python src/train.py --data_dir ./data/behavior_dataset --output_dir ./models

# Or in notebook (cells above)
```

### For Inference:
```python
from src.inference import BehaviorInference

# Use TFLite for mobile
inference = BehaviorInference('./models/behavior_classifier.tflite', tflite_mode=True)

# Predict from image
result = inference.predict_image('./path/to/image.jpg')
print(f"Behavior: {result['prediction']}")
print(f"Confidence: {result['confidence']:.2%}")

# Or real-time from webcam
BehaviorInference.capture_webcam('./models/behavior_classifier.tflite')
```

### For Mobile Deployment:
1. Use the TFLite model: `./models/behavior_classifier.tflite`
2. Integrate with your mobile app framework (React Native, Flutter, etc.)
3. Model size: ~30MB (suitable for mobile)

In [ ]:
print("="*60)
print("REAL-TIME INFERENCE SETUP")
print("="*60)

# Initialize TFLite inference
try:
    tflite_path = './models/behavior_classifier.tflite'
    
    if os.path.exists(tflite_path):
        print(f"✓ TFLite model found: {tflite_path}")
        
        # Test inference speed
        print("\nTesting inference latency...")
        inference = BehaviorInference(tflite_path, class_names=class_names, tflite_mode=True)
        
        # Create dummy image
        dummy_image = np.random.rand(224, 224, 3).astype('float32')
        
        import time
        start = time.time()
        for _ in range(100):
            result = inference.predict(dummy_image)
        elapsed = (time.time() - start) / 100
        
        print(f"  Average inference time: {elapsed*1000:.2f}ms")
        print(f"  Throughput: {1/elapsed:.1f} FPS")
        
        print("\n✓ Model ready for mobile deployment!")
        print(f"  Model size: {os.path.getsize(tflite_path)/1024:.2f} KB")
        print(f"  Classes: {', '.join(class_names)}")
    else:
        print(f"⚠ TFLite model not found: {tflite_path}")
        print("  Please train and save the model first.")
        
except Exception as e:
    print(f"Error during inference setup: {e}")

## 9. Real-time Inference and Mobile Deployment

In [ ]:
if model is not None:
    os.makedirs('./models', exist_ok=True)
    
    # Save Keras model
    model_path = './models/behavior_classifier.h5'
    classifier.save_model(model_path)
    
    # Convert to TFLite
    tflite_path = './models/behavior_classifier.tflite'
    print("\nConverting to TensorFlow Lite for mobile deployment...")
    classifier.convert_to_tflite(tflite_path, quantize=True)
    
    # Compare file sizes
    keras_size = os.path.getsize(model_path) / (1024 * 1024)
    tflite_size = os.path.getsize(tflite_path) / 1024
    
    print(f"\nModel Sizes:")
    print(f"  Keras Model: {keras_size:.2f} MB")
    print(f"  TFLite Model: {tflite_size:.2f} KB")
    print(f"  Compression: {keras_size / (tflite_size/1024):.1f}x smaller")

## 8. Save Model and Convert to TensorFlow Lite

In [ ]:
if history is not None:
    cm = confusion_matrix(val_true_classes, val_pred_classes)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix - Behavior Classification', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

### Confusion Matrix

In [ ]:
if history is not None:
    print("Generating predictions on validation set...")
    val_predictions = model.predict(val_generator, verbose=0)
    val_pred_classes = np.argmax(val_predictions, axis=1)
    val_true_classes = val_generator.classes
    
    # Classification Report
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(val_true_classes, val_pred_classes, 
                               target_names=class_names, digits=4))
    
    # Overall Accuracy
    accuracy = accuracy_score(val_true_classes, val_pred_classes)
    print(f"Overall Accuracy: {accuracy:.4f}")
else:
    print("No model trained yet. Please load data and train the model first.")

## 7. Evaluate Model Performance

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Training', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Training', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 6. Visualize Training History

In [ ]:
if history is not None:
    print("="*60)
    print("Phase 2: Fine-tuning with unfrozen layers (10 epochs)")
    print("="*60)
    
    classifier.unfreeze_base(num_layers=50)
    
    history_finetune = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=10,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✓ Phase 2 training complete")
    
    # Combine histories
    for key in history.history:
        history.history[key].extend(history_finetune.history[key])
else:
    print("Skipping fine-tuning: No data loaded")

### Fine-tuning: Unfreeze base layers

In [ ]:
data_dir = "./data/behavior_dataset"

# Check if data exists
if not os.path.exists(data_dir):
    print(f"⚠ Data directory not found: {data_dir}")
    print("\nTo train the model, create the following structure:")
    print(f"{data_dir}/")
    for class_name in class_names:
        print(f"  └── {class_name}/")
        print(f"      ├── image1.jpg")
        print(f"      ├── image2.jpg")
        print(f"      └── ...")
else:
    print("Loading data...")
    try:
        train_generator, val_generator = data_loader.load_from_directory(data_dir)
        print(f"✓ Data loaded successfully")
        print(f"  Training samples: {train_generator.samples}")
        print(f"  Validation samples: {val_generator.samples}")
        
        # Create callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy',
                patience=3,
                restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=2,
                min_lr=1e-7
            ),
            keras.callbacks.ModelCheckpoint(
                './models/best_model.h5',
                monitor='val_accuracy',
                save_best_only=True
            )
        ]
        
        # Create models directory
        os.makedirs('./models', exist_ok=True)
        
        # Train phase 1: frozen base
        print("\n" + "="*60)
        print("Phase 1: Training with frozen base model (25 epochs)")
        print("="*60)
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=25,
            callbacks=callbacks,
            verbose=1
        )
        
        print("\n✓ Phase 1 training complete")
        
    except Exception as e:
        print(f"Error loading data: {e}")
        history = None

## 5. Load Data and Train Model

This section will train the model. Make sure your dataset is in the correct directory structure.

In [ ]:
print("Building MobileNetV2 Transfer Learning Model...")
model = classifier.build_model(freeze_base=True, dropout_rate=0.5)

print("\nModel Architecture:")
classifier.get_summary()

print(f"\nTotal parameters: {model.count_params():,}")
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

## 4. Build Transfer Learning Model

Using MobileNetV2 as backbone for fast inference on mobile devices.

In [ ]:
batch_size = 32
image_size = (224, 224)

# Initialize model
classifier = BehaviorClassifier(num_classes=num_classes, input_shape=(224, 224, 3))
print("✓ Model initialized")

# Initialize data loader
data_loader = BehaviorDataLoader(
    image_size=image_size, 
    batch_size=batch_size,
    class_names=class_names
)
print("✓ Data loader initialized")

## 3. Initialize Model and Data Loader

In [ ]:
class_names = ['Using Computer', 'Writing', 'Reading', 'Distracted', 'Sleepy']
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")

# Expected data structure:
# data/behavior_dataset/
#   ├── Using Computer/
#   ├── Writing/
#   ├── Reading/
#   ├── Distracted/
#   └── Sleepy/

data_dir = "./data/behavior_dataset"
print(f"\nExpected data directory: {data_dir}")
print("Please organize your dataset with subdirectories for each class.")
print("Each class directory should contain image files (.jpg, .png)")

# Check if data directory exists
if os.path.exists(data_dir):
    print(f"\n✓ Data directory found!")
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        if os.path.exists(class_dir):
            num_images = len([f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.png'))])
            print(f"  {class_name}: {num_images} images")
        else:
            print(f"  ⚠ {class_name}: directory not found")
else:
    print(f"\n⚠ Data directory not found: {data_dir}")

## 2. Load and Explore Dataset

For this demo, we'll create synthetic behavior data or use a sample dataset. In production, replace with your actual behavior dataset.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import sys
from pathlib import Path
import json

# Add src directory to path
sys.path.insert(0, str(Path().resolve() / 'src'))

from model import BehaviorClassifier
from data_loader import BehaviorDataLoader
from inference import BehaviorInference

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set style for visualizations
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Import Required Libraries

# Human Behavior Classification with Transfer Learning

This notebook demonstrates building a real-time human behavior classification model using MobileNetV2 transfer learning. The model classifies 5 behaviors:
- Using Computer
- Writing
- Reading
- Distracted
- Sleepy

**Optimized for mobile deployment with TensorFlow Lite conversion.**